In [1]:
import sys
import os
import json
import torch

BACKEND_PATH = os.path.join(os.getcwd(), "..", "backend")
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

from app.models.nanogpt import NanoGPT
print("Environnement configuré et NanoGPT importé ✓")

Environnement configuré et NanoGPT importé ✓


In [2]:
# --- CELLULE 2 COMPLÈTE ET CORRIGÉE (SPRINT 3) ---
import os
import json
import torch

# 1. Lecture des données
with open("../data/dataset_fr.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }

# MODIFICATION POUR LE BACKEND : On encapsule stoi dans la structure attendue {"stoi": ...}
vocab_package = {"stoi": stoi}

# Sauvegarde immédiate du vocabulaire au format correct dans le backend
weights_dir = os.path.join(BACKEND_PATH, "app", "weights")
os.makedirs(weights_dir, exist_ok=True)
with open(os.path.join(weights_dir, "vocab_fr.json"), "w", encoding="utf-8") as f:
    json.dump(vocab_package, f, ensure_ascii=False, indent=4)

print(f"Vocabulaire FR généré ({vocab_size} caractères) et sauvegardé au format backend ✓")

# 2. Conversion de tout le texte en nombres (tenseurs PyTorch)
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

# 3. Séparation 90% pour l'entraînement et 10% pour la validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Données préparées : train_data ({len(train_data)} tokens) et val_data ({len(val_data)} tokens) créées ! ✓")

Vocabulaire FR généré (32 caractères) et sauvegardé au format backend ✓
Données préparées : train_data (19326 tokens) et val_data (2148 tokens) créées ! ✓


In [3]:
# Initialisation avec la vraie taille du vocabulaire
model_fr_cond = NanoGPT(vocab_size=vocab_size, n_embed=64, n_head=4, n_layer=4, block_size=24)
print("Modèle nanoGPT initialisé avec succès ✓")

Modèle nanoGPT initialisé avec succès ✓


In [4]:
import torch
import torch.optim as optim

# 1. Détection automatique du matériel (GPU vs CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Périphérique d'entraînement détecté : {device.upper()}")

# 2. Configuration stricte des Hyperparamètres d'entraînement
max_iters = 2000      # Nombre d'itérations
eval_interval = 400    # Fréquence d'affichage de la Loss
learning_rate = 1e-3   # Taux d'apprentissage pour AdamW
batch_size = 32        # Taille des sous-ensembles (Mini-batchs)
block_size = 24        # Longueur du contexte de caractères

# 3. Préparation de l'optimiseur (AdamW)
optimizer = optim.AdamW(model_fr_cond.parameters(), lr=learning_rate)

# Fonction utilitaire pour générer des batchs spécifiques au Sprint 3
def get_batch_fr(split):
    data_split = train_data if split == 'train' else val_data
    # Choix d'indices de départ aléatoires
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device) # <--- Maintenant 'device' est bien défini !

print("🚀 Début de l'entraînement du Transformer Français...")
model_fr_cond.to(device) # On s'assure que le modèle est sur le bon périphérique
model_fr_cond.train()    # Mode entraînement (active le dropout)

for iteration in range(max_iters):
    
    # Évaluation régulière pour suivre la convergence
    if iteration % eval_interval == 0 or iteration == max_iters - 1:
        model_fr_cond.eval() # Mode évaluation (désactive le dropout)
        with torch.no_grad():
            xb_t, yb_t = get_batch_fr('train')
            xb_v, yb_v = get_batch_fr('val')
            _, loss_train = model_fr_cond(xb_t, yb_t)
            _, loss_val = model_fr_cond(xb_v, yb_v)
        print(f"Itération {iteration:4d} | Loss Train: {loss_train.item():.4f} | Loss Val: {loss_val.item():.4f}")
        model_fr_cond.train()

    # --- L'ALGORITHME D'APPRENTISSAGE ---
    xb, yb = get_batch_fr('train')
    
    # Forward Pass
    logits, loss = model_fr_cond(xb, yb)
    
    # Backward Pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    
    # Mise à jour des poids
    optimizer.step()

print("\n✓ Entraînement du Sprint 3 terminé avec succès ! Les poids sont stabilisés.")

Périphérique d'entraînement détecté : CPU
🚀 Début de l'entraînement du Transformer Français...
Itération    0 | Loss Train: 3.4498 | Loss Val: 3.4336
Itération  400 | Loss Train: 1.6777 | Loss Val: 1.9954
Itération  800 | Loss Train: 1.0648 | Loss Val: 1.9249
Itération 1200 | Loss Train: 0.7604 | Loss Val: 1.8443
Itération 1600 | Loss Train: 0.7973 | Loss Val: 1.8980
Itération 1999 | Loss Train: 0.6551 | Loss Val: 2.1012

✓ Entraînement du Sprint 3 terminé avec succès ! Les poids sont stabilisés.


In [5]:
# Sauvegarde des poids entraînés
model_path = os.path.join(weights_dir, "model_fr.pt")
torch.save(model_fr_cond.to('cpu').state_dict(), model_path)
print(f"Poids exportés avec succès ({model_path}) ! Prêt pour l'API ✓")

Poids exportés avec succès (c:\Users\HP\Documents\ProjetVersion1\notebooks\..\backend\app\weights\model_fr.pt) ! Prêt pour l'API ✓
